# Country Segmentation for IBRD Portfolio

This notebook aggregates the portfolio by country, standardizes features, tests cluster counts, and segments countries by behavior to guide leadership engagement strategies.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path('/home/rigii/ATA').resolve()]
project_root = next((root for root in candidate_roots if (root / 'data' / 'processed' / 'ibrd_clean.csv').exists()), Path.cwd().resolve())

DATA_PATH = project_root / 'data' / 'processed' / 'ibrd_clean.csv'
FEATURES_DIR = project_root / 'data' / 'features'
MODELS_DIR = project_root / 'models_pickle'
FIGURES_DIR = project_root / 'reports' / 'figures'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} loan records across {df['Country / Economy'].nunique()} countries")


Project root: /home/rigii/ATA
Loaded 9518 loan records across 148 countries


In [2]:
country_df = df.groupby('Country / Economy', as_index=False).agg(
    total_commitments=('Original Principal Amount (US$)', 'sum'),
    total_disbursed=('Disbursed Amount (US$)', 'sum'),
    total_repaid=('Repaid to IBRD (US$)', 'sum'),
    total_outstanding=('Due to IBRD (US$)', 'sum'),
    num_loans=('Loan Number', 'count'),
    avg_loan_size=('Original Principal Amount (US$)', 'mean'),
    avg_repayment_ratio=('repayment_ratio', 'mean'),
    avg_risk_score=('risk_score', 'mean'),
    avg_loan_age=('loan_age_years', 'mean'),
    cancellation_rate=('is_cancelled', 'mean'),
    active_loans_count=('is_active', 'sum'),
)

country_df['cancellation_rate'] = country_df['cancellation_rate'].fillna(0) * 100
for col in ['avg_loan_size', 'avg_repayment_ratio', 'avg_risk_score', 'avg_loan_age']:
    country_df[col] = country_df[col].fillna(0)
country_df['total_outstanding'] = country_df['total_outstanding'].fillna(0)
country_df = country_df.sort_values('total_commitments', ascending=False).reset_index(drop=True)
print(country_df.head().to_string(index=False))


Country / Economy  total_commitments  total_disbursed  total_repaid  total_outstanding  num_loans  avg_loan_size  avg_repayment_ratio  avg_risk_score  avg_loan_age  cancellation_rate  active_loans_count
            India       9.089926e+10     6.151363e+10  3.771143e+10       2.367162e+10        439   2.070598e+08             0.599343        0.307745     26.390150           3.872437                 187
           Brazil       7.601575e+10     5.928965e+10  4.314639e+10       1.609742e+10        530   1.434259e+08             0.740521        0.190094     29.336738           0.754717                 136
        Indonesia       7.244698e+10     5.573187e+10  3.380824e+10       2.183555e+10        657   1.102694e+08             0.770526        0.148858     32.549462           1.217656                 112
          Turkiye       6.391224e+10     4.260069e+10  2.733878e+10       1.515292e+10        372   1.718071e+08             0.713211        0.187634     29.776961           0.268817      

In [3]:
numeric_cols = [
    'total_commitments', 'total_disbursed', 'total_repaid', 'total_outstanding',
    'num_loans', 'avg_loan_size', 'avg_repayment_ratio', 'avg_risk_score', 'avg_loan_age',
    'cancellation_rate', 'active_loans_count'
]

feature_df = country_df[numeric_cols].copy().fillna(0)
log_cols = ['total_commitments', 'total_disbursed', 'total_repaid', 'total_outstanding', 'avg_loan_size', 'num_loans']
for col in log_cols:
    feature_df[col] = np.log1p(feature_df[col].clip(lower=0))

scaler = StandardScaler()
scaled = scaler.fit_transform(feature_df)
scaled_df = pd.DataFrame(scaled, columns=numeric_cols, index=country_df.index)
print('Scaled data shape:', scaled_df.shape)
print(scaled_df.head().to_string(index=False))


Scaled data shape: (148, 11)
 total_commitments  total_disbursed  total_repaid  total_outstanding  num_loans  avg_loan_size  avg_repayment_ratio  avg_risk_score  avg_loan_age  cancellation_rate  active_loans_count
          2.002729         1.472112      1.459491           1.268399   2.001388       1.517460            -0.292568        0.733807     -0.654680           0.179308            5.996372
          1.927644         1.460955      1.498649           1.231312   2.132998       1.207205             0.217086        0.115944     -0.490865          -0.280882            4.233475
          1.907452         1.442205      1.427713           1.260634   2.283131       0.985074             0.325404       -0.100614     -0.312255          -0.212550            3.403876
          1.854818         1.360797      1.365937           1.225497   1.885733       1.359763             0.118495        0.103025     -0.466391          -0.352603            2.781678
          1.843889         1.435725      1.483

In [4]:
inertia = []
silhouette_scores = []
k_values = range(2, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(scaled_df)
    inertia.append(km.inertia_)
    if len(np.unique(labels)) > 1:
        silhouette_scores.append(silhouette_score(scaled_df, labels))
    else:
        silhouette_scores.append(np.nan)

elbow_df = pd.DataFrame({'k': list(k_values), 'inertia': inertia})
sil_df = pd.DataFrame({'k': list(k_values), 'silhouette': silhouette_scores})

fig = make_subplots(rows=1, cols=2, subplot_titles=('Elbow Method', 'Silhouette Scores'))
fig.add_trace(go.Scatter(x=elbow_df['k'], y=elbow_df['inertia'], mode='lines+markers', name='Inertia'), row=1, col=1)
fig.add_trace(go.Scatter(x=sil_df['k'], y=sil_df['silhouette'], mode='lines+markers', name='Silhouette'), row=1, col=2)
fig.update_layout(title='Optimal K Selection for Country Segmentation', template='plotly_white')
fig.write_image(FIGURES_DIR / 'elbow_silhouette.png', width=1400, height=500)
fig.show()
print(elbow_df.to_string(index=False))
print(sil_df.to_string(index=False))


 k     inertia
 2 1071.753229
 3  868.933970
 4  734.138881
 5  619.859334
 6  544.022730
 7  472.286067
 8  425.966804
 9  390.063733
10  355.796708
 k  silhouette
 2    0.346477
 3    0.373241
 4    0.339145
 5    0.349886
 6    0.314176
 7    0.321457
 8    0.301043
 9    0.273772
10    0.265880


In [5]:
best_k = 4
if sil_df['silhouette'].notna().any():
    best_k = int(sil_df.loc[sil_df['silhouette'].idxmax(), 'k'])
print(f"Using k = {best_k} for clustering")

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
country_df['cluster'] = kmeans.fit_predict(scaled_df)
cluster_centers = scaler.inverse_transform(kmeans.cluster_centers_)
center_df = pd.DataFrame(cluster_centers, columns=numeric_cols)
center_df['cluster'] = range(best_k)
print(center_df.round(2).to_string(index=False))


Using k = 3 for clustering
 total_commitments  total_disbursed  total_repaid  total_outstanding  num_loans  avg_loan_size  avg_repayment_ratio  avg_risk_score  avg_loan_age  cancellation_rate  active_loans_count  cluster
             22.53            22.22         21.50              19.21       4.39          18.15                 0.66            0.24         29.13               2.75               27.61        0
             18.62            18.49         18.02               1.68       2.23          16.58                 0.77            0.04         50.77               2.14                0.19        1
             18.02            13.09         11.46              12.96       1.44          16.93                 0.09            0.64          9.72               6.25                3.62        2


In [6]:
cluster_rank = country_df.groupby('cluster').agg(
    avg_commitment=('total_commitments', 'mean'),
    avg_risk=('avg_risk_score', 'mean'),
    active_share=('active_loans_count', 'mean'),
).reset_index()

segment_names = {}
for cluster_id in sorted(cluster_rank['cluster'].unique()):
    row = cluster_rank[cluster_rank['cluster'] == cluster_id].iloc[0]
    if row['avg_risk'] >= cluster_rank['avg_risk'].quantile(0.75):
        segment_names[cluster_id] = 'High Risk Active'
    elif row['avg_commitment'] >= cluster_rank['avg_commitment'].quantile(0.75):
        segment_names[cluster_id] = 'Low Risk Large'
    elif row['active_share'] <= cluster_rank['active_share'].quantile(0.25):
        segment_names[cluster_id] = 'At Risk Dormant'
    else:
        segment_names[cluster_id] = 'Small Stable'

country_df['segment_name'] = country_df['cluster'].map(segment_names)
print(segment_names)

for cluster_id in sorted(country_df['cluster'].unique()):
    segment = country_df[country_df['cluster'] == cluster_id].copy()
    label = segment_names[cluster_id]
    print(f"\nSegment: {label}")
    print(f"Countries: {len(segment)}")
    print(f"Avg commitment: ${segment['total_commitments'].mean():,.0f}")
    print(f"Avg risk score: {segment['avg_risk_score'].mean():.2f}")
    print(f"Avg repayment ratio: {segment['avg_repayment_ratio'].mean():.2%}")
    print('Top 5 countries:')
    print(segment[['Country / Economy', 'total_commitments', 'avg_risk_score', 'avg_repayment_ratio']].sort_values('total_commitments', ascending=False).head(5).to_string(index=False))


{np.int32(0): 'Low Risk Large', np.int32(1): 'At Risk Dormant', np.int32(2): 'High Risk Active'}

Segment: Low Risk Large
Countries: 71
Avg commitment: $13,778,198,931
Avg risk score: 0.24
Avg repayment ratio: 65.88%
Top 5 countries:
Country / Economy  total_commitments  avg_risk_score  avg_repayment_ratio
            India       9.089926e+10        0.307745             0.599343
           Brazil       7.601575e+10        0.190094             0.740521
        Indonesia       7.244698e+10        0.148858             0.770526
          Turkiye       6.391224e+10        0.187634             0.713211
           Mexico       6.227015e+10        0.100391             0.831662

Segment: At Risk Dormant
Countries: 69
Avg commitment: $236,540,656
Avg risk score: 0.04
Avg repayment ratio: 77.07%
Top 5 countries:
Country / Economy  total_commitments  avg_risk_score  avg_repayment_ratio
         Portugal       1338800000.0        0.085714             0.847485
 Papua New Guinea        866620000.0   

In [7]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(scaled_df)
pca_df = pd.DataFrame(pca_coords, columns=['PC1', 'PC2'])
pca_df['cluster'] = country_df['cluster'].values
pca_df['country'] = country_df['Country / Economy'].values

pca_fig = px.scatter(
    pca_df, x='PC1', y='PC2', color='cluster', hover_name='country',
    title='Country Clusters (PCA)', color_continuous_scale='Viridis'
)
pca_fig.write_image(FIGURES_DIR / 'country_clusters_pca.png', width=1000, height=700)
pca_fig.show()


In [8]:
cluster_sizes = country_df['segment_name'].value_counts().reset_index()
cluster_sizes.columns = ['segment_name', 'country_count']
size_fig = px.bar(
    cluster_sizes, x='segment_name', y='country_count', color='segment_name',
    title='Country Segment Sizes', template='plotly_white'
)
size_fig.write_image(FIGURES_DIR / 'cluster_sizes.png', width=900, height=600)
size_fig.show()


In [9]:
cluster_profile = country_df.groupby('segment_name')[numeric_cols].mean().round(3)
cluster_profile = cluster_profile.reindex(['Low Risk Large', 'High Risk Active', 'Small Stable', 'At Risk Dormant'])
heatmap_fig = px.imshow(
    cluster_profile.values, color_continuous_scale='RdBu_r', text_auto=True,
    x=cluster_profile.columns, y=cluster_profile.index, title='Country Cluster Feature Heatmap'
)
heatmap_fig.update_layout(xaxis={'tickangle': 45}, template='plotly_white')
heatmap_fig.write_image(FIGURES_DIR / 'cluster_heatmap.png', width=1400, height=800)
heatmap_fig.show()


In [10]:
output_cols = [
    'Country / Economy', 'cluster', 'segment_name', 'total_commitments', 'total_disbursed', 'total_repaid', 'total_outstanding',
    'num_loans', 'avg_loan_size', 'avg_repayment_ratio', 'avg_risk_score', 'avg_loan_age', 'cancellation_rate', 'active_loans_count'
]
output_df = country_df[output_cols].copy()
output_df.to_csv(FEATURES_DIR / 'country_segments.csv', index=False)
print('Saved country segment file to', FEATURES_DIR / 'country_segments.csv')

import joblib
joblib.dump(kmeans, MODELS_DIR / 'country_kmeans.pkl')
joblib.dump(scaler, MODELS_DIR / 'country_scaler.pkl')
print('Saved KMeans model to', MODELS_DIR / 'country_kmeans.pkl')
print('Saved scaler to', MODELS_DIR / 'country_scaler.pkl')


Saved country segment file to /home/rigii/ATA/data/features/country_segments.csv
Saved KMeans model to /home/rigii/ATA/models_pickle/country_kmeans.pkl
Saved scaler to /home/rigii/ATA/models_pickle/country_scaler.pkl


## Strategic recommendations

### Low Risk Large
- Prioritize proactive relationship management and expand co-financing or blended-finance opportunities with the largest stable markets.
- Use these countries as anchor markets for regional knowledge-sharing and capacity-building programs.
- Maintain regular portfolio reviews to preserve strong repayment performance while capturing new demand.

### High Risk Active
- Increase monitoring intensity and trigger early warning reviews for active exposures with elevated risk profiles.
- Offer structured restructuring or technical assistance before stress escalates into default.
- Tighten underwriting, exposure limits, and follow-up governance in these higher-risk markets.

### Small Stable
- Keep a lighter-touch engagement model focused on continuity and local support.
- Use these markets for pilot programs and targeted advisory services with measured risk appetite.
- Preserve operational efficiency while maintaining stable portfolio exposure levels.

### At Risk Dormant
- Prioritize governance and credit-performance reviews to determine whether dormant exposures require remediation.
- Re-engage stakeholders through restructuring, borrower support, or risk reduction measures where needed.
- Focus management attention on underperforming portfolios where cancellation and repayment issues remain elevated.


In [11]:
print('Country segmentation notebook completed successfully.')


Country segmentation notebook completed successfully.
